In [17]:
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *
import pandas as pd

In [18]:


valor_total = 500000

In [19]:
df = pd.read_excel('app14.xlsx')

In [20]:
df.set_index('acao', inplace=True)

In [21]:
df.columns

Index(['categoria', 'risco', 'rentabilidade'], dtype='object')

In [22]:
df.categoria.unique()

array(['Banco', 'Siderurgia', 'Explora'], dtype=object)

In [23]:
df[df['categoria'] == 'Banco']['rentabilidade']

acao
BBAS3    1.10
BBDC4    1.12
ITAU4    1.10
Name: rentabilidade, dtype: float64

In [24]:
df

,categoria,risco,rentabilidade
acao,,,
BBAS3,Banco,Baixo,1.10
BBDC4,Banco,Alto,1.12
CSNA3,Siderurgia,Baixo,1.05
GGBR4,Siderurgia,Alto,1.08
ITAU4,Banco,Alto,1.10
PETR4,Explora,Baixo,1.12
USIM4,Siderurgia,Alto,1.05
VALE5,Explora,Baixo,1.10


In [25]:
df.loc[df['categoria'] == 'Banco']['rentabilidade']

acao
BBAS3    1.10
BBDC4    1.12
ITAU4    1.10
Name: rentabilidade, dtype: float64

In [26]:
acoes = df.index.tolist()
categorias = df['categoria'].unique().tolist()
classificacao_de_risco = df['risco'].unique().tolist()

In [27]:
import numpy as np

In [28]:
model = pyo.ConcreteModel()

model.acoes = pyo.Set(initialize=acoes)
model.categorias = pyo.Set(initialize=categorias)   
model.risco = pyo.Set(initialize=classificacao_de_risco)

model.x = pyo.Var(model.acoes, domain=pyo.NonNegativeReals)
# x é quantidade investida em cada açãoo

def objetivo(model):
    return sum(df.loc[acao]['rentabilidade'] * model.x[acao] for acao in model.acoes)
model.obj = pyo.Objective(rule=objetivo, sense=pyo.maximize)    

#restrições -----------------

#investimento isolado <= 15%
def investimento_isolado_maximo(model, acao):
    return model.x[acao] <= 0.15 * valor_total
model.investimento_isolado = pyo.Constraint(model.acoes, rule=investimento_isolado_maximo)

#investimento isolado minimo >= 5%
def investimento_isolado_minimo(model, acao):
    return model.x[acao] >= 0.05 * valor_total  
model.investimento_isolado_minimo = pyo.Constraint(model.acoes, rule=investimento_isolado_minimo)

#investimento total <= valor_total
def investimento_total(model):
    return sum(model.x[acao] for acao in model.acoes) <= valor_total
model.investimento_total = pyo.Constraint(rule=investimento_total)

#investimento por categoria <= 40%
def investimento_categoria(model, categoria):
    return sum(model.x[acao] for acao in model.acoes if df.loc[acao]['categoria'] == categoria) <= 0.40 * valor_total   
model.investimento_categoria = pyo.Constraint(model.categorias, rule=investimento_categoria)

#Alto risco nao pode ser maior que baixo risco
def risco_alto_baixo(model):
    valor_alto = sum(model.x[acao] for acao in model.acoes if df.loc[acao]['risco'] == 'Alto')
    valor_baixo = sum(model.x[acao] for acao in model.acoes if df.loc[acao]['risco'] == 'Baixo')
    return valor_alto <= valor_baixo
model.risco_alto_baixo = pyo.Constraint(rule=risco_alto_baixo)

In [29]:
model.pprint()

3 Set Declarations
    acoes : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    8 : {'BBAS3', 'BBDC4', 'CSNA3', 'GGBR4', 'ITAU4', 'PETR4', 'USIM4', 'VALE5'}
    categorias : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'Banco', 'Siderurgia', 'Explora'}
    risco : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'Baixo', 'Alto'}

1 Var Declarations
    x : Size=8, Index=acoes
        Key   : Lower : Value : Upper : Fixed : Stale : Domain
        BBAS3 :     0 :  None :  None : False :  True : NonNegativeReals
        BBDC4 :     0 :  None :  None : False :  True : NonNegativeReals
        CSNA3 :     0 :  None :  None : False :  True : NonNegativeReals
        GGBR4 :     0 :  None :  None : False :  True : NonNegativeReals
        ITAU4 :     0 :  None :  None :

In [30]:
opt = SolverFactory('cplex')
res=opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpodht319z.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmp9mf2nygo.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmp9mf2nygo.pyomo.lp
Objective sense      : Maximize
Variables            :       8
Objective nonzeros   :       8
Linear constraints   :      21  [Less: 13,  Greater: 8]
  Nonzeros           :      40
  RHS nonzeros       :      20

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 1.050000         Max   

In [31]:
for m in model.x:
    print(f'{m} = {model.x[m].value}')

print(f"Retorno Investimento: {pyo.value(model.obj)}")

BBAS3 = 75000.0
BBDC4 = 75000.0
CSNA3 = 50000.0
GGBR4 = 75000.0
ITAU4 = 50000.0
PETR4 = 75000.0
USIM4 = 25000.0
VALE5 = 75000.0
Retorno Investimento: 547750.0
